In [2]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if (text.startswith('/Users/') or text.startswith('/home/') or ':\\' in text) and '.' not in name:
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [3]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from bisect import bisect_left
import pickle


In [4]:
gene_exp = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Expression_Trimmed.csv')
gene_exp.name = 'gene_exp'

copy_num = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Copy_Number_Trimmed.csv')
copy_num.name = 'copy_num'

shRNA = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'shRNA_Trimmed.csv')
shRNA.name = 'shRNA'

gene_mut = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Mutation_Trimmed.csv')
gene_mut.name = 'gene_mut'

CRISPR = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'CRISPR_Trimmed.csv')
CRISPR.name = 'CRISPR'


In [5]:
biogrid_pairs = (
    pd.read_csv(DATA_DIR / 'clean_biogrid_interactions.csv')
    .rename(columns={'Gene_A': 'Gene1', 'Gene_B': 'Gene2'})
)
biogrid_pairs['Gene1'] = biogrid_pairs['Gene1'].astype(str).str.strip()
biogrid_pairs['Gene2'] = biogrid_pairs['Gene2'].astype(str).str.strip()
biogrid_pairs = biogrid_pairs.dropna(subset=['Gene1', 'Gene2']).reset_index(drop=True)


In [6]:
genes = sorted(set(gene_exp.iloc[:,0].str.strip()) & 
               set(copy_num.iloc[:,0].str.strip()) & 
               set(shRNA.iloc[:,0].str.strip()) & 
               set(CRISPR.iloc[:,0].str.strip()))
genes = [i.strip() for i in genes]


In [7]:
datasets = gene_exp, copy_num, shRNA, gene_mut, CRISPR
points = density_centers(gene_exp, 7), [0,1,2,3,4,6,8], density_centers(shRNA, 7), [0, 1], density_centers(CRISPR, 7)
cont = True, False, True, False, True


In [ ]:
# a = 'CCND1.CCNB1 E2F3.E2F2 ORC6.MCM3 ORC3.MCM5 ANAPC10.ANAPC3 ANAPC9.ANAPC4 ANAPC11.ANAPC5 CDC20.ANAPC8 MRE11.NBN FANCL.FANCE'
# x = []
# y = []
# for pair in a.split():
#     g1, g2 = pair.split('.')
#     x.append(g1)
#     y.append(g2)

# biogrid_pairs = pd.DataFrame({'Gene1' : x, 'Gene2' : y})
# biogrid_pairs

,Gene1,Gene2
0,CCND1,CCNB1
1,E2F3,E2F2
2,ORC6,MCM3
3,ORC3,MCM5
4,ANAPC10,ANAPC3
5,ANAPC9,ANAPC4
6,ANAPC11,ANAPC5
7,CDC20,ANAPC8
8,MRE11,NBN
9,FANCL,FANCE


In [12]:
temp_time = time.time()
pair_mask = biogrid_pairs['Gene1'].isin(genes) & biogrid_pairs['Gene2'].isin(genes)
pair_table = biogrid_pairs.loc[pair_mask].reset_index(drop=True)
pair_matrix = build_density_map(
    datasets,
    pair_table[['Gene1', 'Gene2']].to_numpy().tolist(),
    points,
    cont,
)
pair_matrix = pd.concat([pair_table, pair_matrix], axis=1)
print(f"converted {len(pair_table)} pairs in {time.time() - temp_time:.2f}s")


converted 6 pairs in 7.95s


In [ ]:
output_path = ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
pair_matrix.to_csv(output_path, index=False)
output_path


PosixPath('/Users/jzx/Documents/Computer Science/PDM_Learn/artifacts/results/clean_biogrid_interactions_pdm_missing.csv')

In [9]:
df = pd.concat([pd.read_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm.csv'), 
                pd.read_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_missing.csv')], axis=0, ignore_index=True)
df

,Gene1,Gene2,Interaction_Type,Confidence_Score,pair,gene_exp.gene_exp.0,gene_exp.gene_exp.1,gene_exp.gene_exp.2,gene_exp.gene_exp.3,gene_exp.gene_exp.4,...,CRISPR.CRISPR.39,CRISPR.CRISPR.40,CRISPR.CRISPR.41,CRISPR.CRISPR.42,CRISPR.CRISPR.43,CRISPR.CRISPR.44,CRISPR.CRISPR.45,CRISPR.CRISPR.46,CRISPR.CRISPR.47,CRISPR.CRISPR.48
0,MAP2K4,FLNC,direct interaction,NaN,MAP2K4.FLNC,-7.174697,-7.173454,-7.154708,-7.105875,-7.161410,...,-5.686287,-6.483615,-6.670788,-6.672022,-6.668549,-6.636919,-6.526064,-6.306870,-6.570739,-6.671530
1,GATA2,PML,direct interaction,NaN,GATA2.PML,-7.174724,-7.174582,-7.173217,-7.172881,-7.173199,...,-5.656049,-6.546896,-6.671619,-6.672032,-6.671773,-6.664401,-6.642069,-6.623849,-6.666457,-6.672020
2,RPA2,STAT3,direct interaction,NaN,RPA2.STAT3,-7.174701,-7.170263,-7.113910,-6.886640,-7.118832,...,-5.235949,-6.266843,-6.659840,-6.671994,-6.661821,-6.314082,-6.310918,-6.555979,-6.650990,-6.671916
3,ARF1,GGA3,direct interaction,NaN,ARF1.GGA3,-7.174668,-7.170583,-7.166090,-7.169137,-7.174248,...,-5.748181,-6.438846,-6.645946,-6.671992,-6.667723,-6.383032,-6.171960,-6.645068,-6.671532,-6.671986
4,ARF3,ARFIP2,direct interaction,NaN,ARF3.ARFIP2,-7.174679,-7.171916,-7.138842,-6.930789,-7.133416,...,-5.497347,-6.187912,-6.661641,-6.672033,-6.671995,-6.670678,-6.642824,-6.455816,-6.616414,-6.671438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84943,E2F3,E2F2,NaN,NaN,E2F3.E2F2,-7.174724,-7.174410,-7.164250,-7.156599,-7.173132,...,-5.651592,-6.433048,-6.649171,-6.671723,-6.670686,-6.664198,-6.655401,-6.641676,-6.669836,-6.671999
84944,ORC6,MCM3,NaN,NaN,ORC6.MCM3,-7.173981,-7.115738,-7.034862,-7.142920,-7.173983,...,-4.940430,-5.319969,-5.745116,-6.672013,-6.671543,-6.622043,-6.416908,-6.496994,-6.197510,-6.342760
84945,ORC3,MCM5,NaN,NaN,ORC3.MCM5,-7.174696,-7.168395,-7.104007,-7.101841,-7.170111,...,-5.130974,-6.300613,-6.666152,-6.672029,-6.670665,-6.615009,-6.304959,-6.318459,-6.648598,-6.671949
84946,ANAPC11,ANAPC5,NaN,NaN,ANAPC11.ANAPC5,-7.174720,-7.174388,-7.174024,-7.174431,-7.174688,...,-5.278657,-6.184865,-6.654270,-6.670849,-6.573591,-6.422391,-6.418285,-6.582323,-6.660632,-6.671882


In [10]:
df.to_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_combined.csv', index=False)

In [12]:
pair_matrix.head()


,Gene1,Gene2,Interaction_Type,Confidence_Score,pair,gene_exp.gene_exp.0,gene_exp.gene_exp.1,gene_exp.gene_exp.2,gene_exp.gene_exp.3,gene_exp.gene_exp.4,...,CRISPR.CRISPR.39,CRISPR.CRISPR.40,CRISPR.CRISPR.41,CRISPR.CRISPR.42,CRISPR.CRISPR.43,CRISPR.CRISPR.44,CRISPR.CRISPR.45,CRISPR.CRISPR.46,CRISPR.CRISPR.47,CRISPR.CRISPR.48
0,MAP2K4,FLNC,direct interaction,NaN,MAP2K4.FLNC,-7.174697,-7.173454,-7.154708,-7.105875,-7.161410,...,-5.686287,-6.483615,-6.670788,-6.672022,-6.668549,-6.636919,-6.526064,-6.306870,-6.570739,-6.671530
1,GATA2,PML,direct interaction,NaN,GATA2.PML,-7.174724,-7.174582,-7.173217,-7.172881,-7.173199,...,-5.656049,-6.546896,-6.671619,-6.672032,-6.671773,-6.664401,-6.642069,-6.623849,-6.666457,-6.672020
2,RPA2,STAT3,direct interaction,NaN,RPA2.STAT3,-7.174701,-7.170263,-7.113910,-6.886640,-7.118832,...,-5.235949,-6.266843,-6.659840,-6.671994,-6.661821,-6.314082,-6.310918,-6.555979,-6.650990,-6.671916
3,ARF1,GGA3,direct interaction,NaN,ARF1.GGA3,-7.174668,-7.170583,-7.166090,-7.169137,-7.174248,...,-5.748181,-6.438846,-6.645946,-6.671992,-6.667723,-6.383032,-6.171960,-6.645068,-6.671532,-6.671986
4,ARF3,ARFIP2,direct interaction,NaN,ARF3.ARFIP2,-7.174679,-7.171916,-7.138842,-6.930789,-7.133416,...,-5.497347,-6.187912,-6.661641,-6.672033,-6.671995,-6.670678,-6.642824,-6.455816,-6.616414,-6.671438


In [10]:
pair_matrix.shape


(84942, 905)